# Inference Observability

Monitoring an inference endpoint looks like monitoring any other HTTP service until it
isn't. The differences are specific and they break the standard playbook:

- **The mean is useless and the p99 is where your users live.** Inference latency
  distributions are heavily right-tailed — queueing, batch formation, variable output
  length — so an average hides everything that matters.
- **"Latency" is not one number.** For a streaming LLM endpoint, time-to-first-token and
  time-per-output-token are separate quantities with separate causes, and a single
  end-to-end timer conflates them.
- **The most common aggregation people apply to percentiles is invalid.** Averaging p99
  across replicas is not the p99, and the error is not small.

This notebook is about getting those three right. It sits alongside
[Inference](inference.ipynb) (the serving techniques themselves),
[DCGM Exporter](../nvidia-gpu-components/dcgm-exporter.ipynb) (GPU metrics) and the
[model serving libraries](../model-serving-libraries/vllm.ipynb).

## What & Why

Observability is the ability to answer questions about production behaviour you did not
anticipate when you deployed. For inference specifically, the questions that come up at
3am are:

- Is it slow because of the model, or because requests are queueing?
- Is the GPU saturated, or idle and waiting on the tokeniser?
- Did latency regress, or did the *input distribution* change (longer prompts)?
- Are we dropping requests, or silently returning degraded output?

None of these are answerable from a single "request duration" metric, and the reason is
that an inference server's latency is a **composition** — queue wait, plus batch
formation, plus prefill, plus decode — and only the decomposition tells you where to
act. A single number tells you that you have a problem, not which one.

The RED method (Rate, Errors, Duration) and USE method (Utilisation, Saturation, Errors)
are the standard scaffolding; the value here is knowing what to put in each slot for a
model server, and what the collection mechanism will and will not let you compute.

## Mental Model

**A restaurant kitchen, and the difference between a busy chef and a backed-up pass.**

A request's life has three distinct stages, and conflating them is the root of most bad
inference dashboards:

- **The order sits on the rail** — *queue time*. Nothing is cooking. This is pure
  capacity: more cooks (replicas) fixes it, a faster cook barely helps.
- **Prep** — *prefill*. Proportional to how elaborate the order is (the prompt length),
  and it happens once.
- **Plating, one item at a time** — *decode*. Proportional to how many items were
  ordered (output tokens), at a fixed rate per item.

A single "how long from order to table" timer mixes all three, so a rise in it is
unattributable: it could be a rush (queue), unusually elaborate orders (prefill), or
larger orders (decode). Only the first is something you fix with more hardware.

The second half of the model is **utilisation versus saturation**. A chef chopping
continuously is at 100% utilisation, and that is fine — it is exactly what you are paying
for. What tells you the kitchen is in trouble is the *rail getting longer*. GPU
utilisation is the chef; queue depth and KV-cache occupancy are the rail. Alert on the
rail.

## Key Concepts

| Term | What it means |
|---|---|
| **RED** | Rate, Errors, Duration — the per-request view. What your users experience. |
| **USE** | Utilisation, Saturation, Errors — the per-resource view. What your hardware is doing. |
| **Counter / Gauge / Histogram** | Prometheus metric types. Counters only go up (rate them), gauges go both ways, histograms bucket observations. |
| **Histogram vs Summary** | Histograms bucket server-side and are **aggregatable across instances**. Summaries compute quantiles per-instance and are **not**. This distinction decides your whole design. |
| **Bucket boundaries** | The fixed cut-points of a histogram. Quantiles are interpolated within a bucket, so boundary choice *is* your precision. |
| **TTFT** | Time To First Token — queue + prefill. Dominated by prompt length and load. |
| **TPOT / ITL** | Time Per Output Token (inter-token latency) — the decode phase. Dominated by batch size and memory bandwidth. |
| **Queue time** | How long a request waited before compute began. The first thing to instrument and the most commonly missing. |
| **Cardinality** | The number of distinct label combinations. Each one is a separate time series; this is how monitoring bills explode. |
| **Exemplars** | Trace IDs attached to histogram buckets, linking a slow bucket to an actual slow request. |
| **KV-cache utilisation** | For LLM servers, the real saturation signal — closer to "how full is the ship" than GPU utilisation is. |
| **Continuous batching metrics** | Running vs waiting request counts, preemptions. The internal state of a modern LLM server. |

## Setup

The examples use only the standard library, so the quantile arithmetic is visible rather
than delegated. The final example shows the `prometheus_client` + FastAPI wiring and is
gated on the package being installed.

In [1]:
# %pip install prometheus-client fastapi uvicorn        # for the last example only

import random
from bisect import bisect_left

random.seed(0)

try:
    import prometheus_client  # noqa: F401
    HAVE_PROM = True
except ImportError:
    HAVE_PROM = False
print("prometheus_client available:", HAVE_PROM)

prometheus_client available: True


## Worked Examples

### Example 1 — the mean is a liar

A realistic inference latency distribution: most requests are fast, a minority queue
behind a full batch. Compare what each summary statistic tells you.

In [2]:
def simulate_latencies(n=20000):
    '''A bimodal, right-tailed distribution: fast path plus queued path.'''
    out = []
    for _ in range(n):
        if random.random() < 0.90:
            out.append(random.gauss(45, 8))            # served from a warm batch
        else:
            out.append(random.gauss(400, 180))         # queued behind a full batch
    return [max(1.0, v) for v in out]

lat = sorted(simulate_latencies())

def pct(sorted_vals, q):
    return sorted_vals[min(len(sorted_vals) - 1, int(q * len(sorted_vals)))]

mean = sum(lat) / len(lat)
print(f"mean   {mean:8.1f} ms")
for q in (0.50, 0.90, 0.95, 0.99, 0.999):
    print(f"p{q*100:<5g} {pct(lat, q):8.1f} ms")
print(f"max    {lat[-1]:8.1f} ms")

worse_than_mean = sum(1 for v in lat if v > mean) / len(lat)
print(f"\nOnly {worse_than_mean:.1%} of requests are slower than the mean, so the mean")
print("describes almost nobody. It sits in the empty gap between the two modes.")
print(f"\nThe p99 is {pct(lat, 0.99)/mean:.1f}x the mean. If you alert on the mean you")
print("will not notice the queueing mode appearing until it dominates.")

mean       80.1 ms
p50        46.1 ms
p90        65.9 ms
p95       394.8 ms
p99       633.2 ms
p99.9     816.4 ms
max      1004.5 ms

Only 9.5% of requests are slower than the mean, so the mean
describes almost nobody. It sits in the empty gap between the two modes.

The p99 is 7.9x the mean. If you alert on the mean you
will not notice the queueing mode appearing until it dominates.


### Example 2 — you cannot average percentiles

This is the single most common error in dashboards, and it is worth seeing the size of.
Three replicas with different load; the true fleet p99 versus the average of the
per-replica p99s.

In [3]:
def replica(n, fast_ms, slow_ms, slow_frac):
    return [max(1.0, random.gauss(slow_ms, slow_ms * 0.4)) if random.random() < slow_frac
            else max(1.0, random.gauss(fast_ms, fast_ms * 0.2)) for _ in range(n)]

replicas = {
    "replica-a (light load)": replica(10000, 40, 200, 0.01),
    "replica-b (light load)": replica(10000, 42, 210, 0.01),
    "replica-c (HOT)":        replica(10000, 90, 900, 0.25),
}

per_replica_p99 = {}
for name, vals in replicas.items():
    per_replica_p99[name] = pct(sorted(vals), 0.99)
    print(f"{name:24} p99 = {per_replica_p99[name]:7.1f} ms")

avg_of_p99 = sum(per_replica_p99.values()) / len(per_replica_p99)
true_p99 = pct(sorted(v for vals in replicas.values() for v in vals), 0.99)

print(f"\naverage of the p99s : {avg_of_p99:7.1f} ms   <- what a dashboard usually shows")
print(f"true fleet p99      : {true_p99:7.1f} ms   <- what users actually experience")
print(f"error               : {100*(avg_of_p99 - true_p99)/true_p99:+.1f}%")

print("\nThe average is pulled down by the two healthy replicas and understates the")
print("real tail. It can also OVERstate it -- the direction depends on the mix, which")
print("is exactly why the operation is meaningless rather than merely biased.")
print("\nThe fix is structural: export a HISTOGRAM, sum the buckets across replicas,")
print("and compute the quantile from the summed buckets. That is what")
print("histogram_quantile(0.99, sum(rate(..._bucket[5m])) by (le)) does in PromQL,")
print("and it is why Prometheus histograms beat summaries for anything replicated.")

replica-a (light load)   p99 =    63.9 ms
replica-b (light load)   p99 =    81.2 ms
replica-c (HOT)          p99 =  1518.1 ms

average of the p99s :   554.4 ms   <- what a dashboard usually shows
true fleet p99      :  1303.8 ms   <- what users actually experience
error               : -57.5%

The average is pulled down by the two healthy replicas and understates the
real tail. It can also OVERstate it -- the direction depends on the mix, which
is exactly why the operation is meaningless rather than merely biased.

The fix is structural: export a HISTOGRAM, sum the buckets across replicas,
and compute the quantile from the summed buckets. That is what
histogram_quantile(0.99, sum(rate(..._bucket[5m])) by (le)) does in PromQL,
and it is why Prometheus histograms beat summaries for anything replicated.


### Example 3 — histograms are lossy, and the buckets decide how lossy

`histogram_quantile` interpolates linearly *within* a bucket. If your p99 falls in a
bucket spanning 250ms–500ms, your p99 is accurate to ±125ms. Bucket layout is not a
detail.

In [4]:
DEFAULT_BUCKETS = [.005, .01, .025, .05, .075, .1, .25, .5, .75, 1, 2.5, 5, 7.5, 10]
TUNED_BUCKETS = [.02, .03, .04, .05, .06, .08, .15, .25, .35, .45, .6, .8, 1.0, 1.5]

def to_histogram(values_s, bounds):
    counts = [0] * (len(bounds) + 1)
    for v in values_s:
        counts[bisect_left(bounds, v)] += 1
    return counts

def histogram_quantile(counts, bounds, q):
    '''The same linear interpolation Prometheus performs.'''
    total = sum(counts)
    target = q * total
    cum = 0
    for i, c in enumerate(counts):
        if cum + c >= target:
            lo = bounds[i - 1] if i > 0 else 0.0
            hi = bounds[i] if i < len(bounds) else bounds[-1]
            frac = (target - cum) / c if c else 0
            return lo + (hi - lo) * frac
        cum += c
    return bounds[-1]

lat_s = sorted(v / 1000.0 for v in lat)          # the Example 1 data, in seconds
exact = {q: pct(lat_s, q) for q in (0.50, 0.90, 0.95, 0.99)}

print(f"{'quantile':>9} {'exact':>9} {'default buckets':>18} {'tuned buckets':>16}")
for q in (0.50, 0.90, 0.95, 0.99):
    d = histogram_quantile(to_histogram(lat_s, DEFAULT_BUCKETS), DEFAULT_BUCKETS, q)
    t = histogram_quantile(to_histogram(lat_s, TUNED_BUCKETS), TUNED_BUCKETS, q)
    print(f"p{q*100:<8g} {exact[q]*1000:8.1f}ms {d*1000:15.1f}ms "
          f"({100*(d-exact[q])/exact[q]:+5.1f}%) {t*1000:9.1f}ms "
          f"({100*(t-exact[q])/exact[q]:+5.1f}%)")

print("\nRead this honestly: tuning the buckets helps at p50 and p99 and makes p90")
print("WORSE. There is no bucket set that is better everywhere -- you are choosing")
print("where to spend a fixed resolution budget.")
print("\np90 is hard here for a reason no bucketing can fix: it falls at ~66ms, in the")
print("empty gap between the fast mode (~45ms) and the queued mode (~400ms). Almost")
print("no samples land near it, so a bucket spanning the gap must interpolate across")
print("a region with no data. A quantile in a sparse region is imprecise, full stop.")
print("\nRule of thumb: put boundaries tightly around the quantiles you actually alert")
print("on, accept coarse resolution elsewhere, and do not quote a quantile that sits")
print("in a gap. Measure the distribution first, then choose.")

 quantile     exact    default buckets    tuned buckets
p50           46.1ms            43.7ms ( -5.1%)      46.1ms ( +0.1%)
p90           65.9ms            74.6ms (+13.2%)      76.8ms (+16.7%)
p95          394.8ms           390.0ms ( -1.2%)     391.8ms ( -0.8%)
p99          633.2ms           676.4ms ( +6.8%)     660.9ms ( +4.4%)

Read this honestly: tuning the buckets helps at p50 and p99 and makes p90
WORSE. There is no bucket set that is better everywhere -- you are choosing
where to spend a fixed resolution budget.

p90 is hard here for a reason no bucketing can fix: it falls at ~66ms, in the
empty gap between the fast mode (~45ms) and the queued mode (~400ms). Almost
no samples land near it, so a bucket spanning the gap must interpolate across
a region with no data. A quantile in a sparse region is imprecise, full stop.

Rule of thumb: put boundaries tightly around the quantiles you actually alert
on, accept coarse resolution elsewhere, and do not quote a quantile that sits
in a g

### Example 4 — decomposing LLM latency, and instrumenting it

End-to-end duration is the sum of parts with different causes. Instrument the parts.

In [5]:
def simulate_llm_request(rho):
    '''One request at server utilisation `rho` (0-1).

    Queue time follows the standard queueing blow-up ~ rho/(1-rho); compute time grows
    only mildly, because a larger batch is more efficient per request even as it is
    slower per token.
    '''
    prompt_tokens = int(random.lognormvariate(6.0, 0.7))       # heavy right tail
    output_tokens = int(random.lognormvariate(4.5, 0.8)) + 1
    batch = 1 + 2 * rho
    queue_ms = random.expovariate(1 / (8 * rho / (1 - rho)))
    prefill_ms = 5 + prompt_tokens * 0.05 * (1 + 0.3 * batch)  # scales with PROMPT
    tpot_ms = 12 * (1 + 0.15 * batch)                          # per output token
    return dict(prompt_tokens=prompt_tokens, output_tokens=output_tokens,
                queue_ms=queue_ms, prefill_ms=prefill_ms, tpot_ms=tpot_ms,
                ttft_ms=queue_ms + prefill_ms,
                total_ms=queue_ms + prefill_ms + output_tokens * tpot_ms)

KEYS = ("queue_ms", "prefill_ms", "tpot_ms", "ttft_ms", "total_ms")
print(f"{'utilisation':>13} | " + " ".join(f"{k[:-3]:>10}" for k in KEYS) + "   (p50)")
baseline = {}
for rho, label in ((0.50, "0.50 quiet"), (0.80, "0.80 busy"), (0.92, "0.92 near cap")):
    reqs = [simulate_llm_request(rho) for _ in range(4000)]
    med = {k: pct(sorted(r[k] for r in reqs), 0.5) for k in KEYS}
    baseline = baseline or med
    print(f"{label:>13} | " + " ".join(f"{med[k]:10.1f}" for k in KEYS))
    if rho == 0.92:
        print(f"{'growth vs 0.50':>13} | " +
              " ".join(f"{med[k]/baseline[k]:9.1f}x" for k in KEYS))

print("\nThis is the decomposition earning its keep. From comfortable to near-capacity:")
print("  queue time grows by an order of magnitude  <- the whole problem")
print("  prefill and TPOT grow ~1.1x               <- compute is barely affected")
print("\nIf you export only total_ms you see a modest rise and no idea why. The fix")
print("for a queue problem (more replicas) and the fix for a compute problem (faster")
print("kernels, better quantisation) are completely different and quite expensive to")
print("get wrong.")
print("\nSecond lesson, from the total_ms column: it is dominated by OUTPUT LENGTH.")
print("A p99 total that moved because users asked longer questions is not a")
print("regression. TPOT is the load-sensitive signal; total_ms is demand-sensitive.")

  utilisation |      queue    prefill       tpot       ttft      total   (p50)
   0.50 quiet |        5.8       36.7       15.6       45.3     1452.4
    0.80 busy |       22.5       41.5       16.7       71.8     1556.9
0.92 near cap |       63.3       42.0       17.1      118.3     1689.9
growth vs 0.50 |      10.9x       1.1x       1.1x       2.6x       1.2x

This is the decomposition earning its keep. From comfortable to near-capacity:
  queue time grows by an order of magnitude  <- the whole problem
  prefill and TPOT grow ~1.1x               <- compute is barely affected

If you export only total_ms you see a modest rise and no idea why. The fix
for a queue problem (more replicas) and the fix for a compute problem (faster
kernels, better quantisation) are completely different and quite expensive to
get wrong.

Second lesson, from the total_ms column: it is dominated by OUTPUT LENGTH.
A p99 total that moved because users asked longer questions is not a
regression. TPOT is the load

In [6]:
INSTRUMENTATION = '''
from prometheus_client import Counter, Histogram, Gauge, make_asgi_app
from fastapi import FastAPI
import time

# Buckets chosen for THIS workload (Example 3), not the library defaults.
TTFT = Histogram("inference_ttft_seconds", "Time to first token",
                 ["model"], buckets=[.01,.02,.05,.1,.2,.3,.5,.8,1.2,2,3,5])
TPOT = Histogram("inference_tpot_seconds", "Time per output token",
                 ["model"], buckets=[.005,.01,.02,.03,.05,.08,.12,.2])
QUEUE = Histogram("inference_queue_seconds", "Time waiting before compute",
                  ["model"], buckets=[.001,.005,.01,.05,.1,.5,1,5])
REQS = Counter("inference_requests_total", "Requests", ["model", "status"])
TOKENS = Counter("inference_tokens_total", "Tokens", ["model", "kind"])
RUNNING = Gauge("inference_running_requests", "In-flight requests", ["model"])
KV = Gauge("inference_kv_cache_utilisation", "KV-cache used fraction", ["model"])

app = FastAPI()
app.mount("/metrics", make_asgi_app())          # Prometheus scrapes this

@app.post("/generate")
async def generate(req: dict):
    model = req["model"]                        # BOUNDED label -- never a user string
    RUNNING.labels(model).inc()
    enqueued = time.perf_counter()
    try:
        started = await scheduler.acquire()
        QUEUE.labels(model).observe(started - enqueued)

        first_token_at = None
        n_out = 0
        async for _tok in engine.stream(req):
            if first_token_at is None:
                first_token_at = time.perf_counter()
                TTFT.labels(model).observe(first_token_at - enqueued)   # includes queue
            n_out += 1

        if n_out > 1:
            TPOT.labels(model).observe(
                (time.perf_counter() - first_token_at) / (n_out - 1))
        TOKENS.labels(model, "output").inc(n_out)
        TOKENS.labels(model, "prompt").inc(req["prompt_tokens"])
        REQS.labels(model, "ok").inc()
    except Exception:
        REQS.labels(model, "error").inc()
        raise
    finally:
        RUNNING.labels(model).dec()
'''

PROMQL = '''
# p99 TTFT across the whole fleet -- sum buckets FIRST, then take the quantile
histogram_quantile(0.99, sum(rate(inference_ttft_seconds_bucket[5m])) by (le, model))

# throughput, the metric that actually matters for cost
sum(rate(inference_tokens_total{kind="output"}[5m])) by (model)

# error ratio (RED)
sum(rate(inference_requests_total{status="error"}[5m]))
  / sum(rate(inference_requests_total[5m]))

# saturation: queueing is the leading indicator, KV-cache is the capacity ceiling
histogram_quantile(0.9, sum(rate(inference_queue_seconds_bucket[5m])) by (le))
max(inference_kv_cache_utilisation) by (model)

# GPU utilisation from DCGM -- necessary, and on its own not sufficient
avg(DCGM_FI_DEV_GPU_UTIL) by (gpu)
'''

print("--- instrumentation ---")
print(INSTRUMENTATION)
print("--- the PromQL that consumes it ---")
print(PROMQL)

if HAVE_PROM:
    from prometheus_client import Histogram, CollectorRegistry, generate_latest
    reg = CollectorRegistry()
    h = Histogram("demo_ttft_seconds", "demo", buckets=[.05, .1, .5, 1, 2], registry=reg)
    for v in (0.03, 0.07, 0.2, 0.9, 3.0):
        h.observe(v)
    lines = [l for l in generate_latest(reg).decode().splitlines()
             if "bucket" in l or l.endswith("_count")]
    print("--- what a scrape of that histogram looks like ---")
    print("\n".join(lines))

--- instrumentation ---

from prometheus_client import Counter, Histogram, Gauge, make_asgi_app
from fastapi import FastAPI
import time

# Buckets chosen for THIS workload (Example 3), not the library defaults.
TTFT = Histogram("inference_ttft_seconds", "Time to first token",
                 ["model"], buckets=[.01,.02,.05,.1,.2,.3,.5,.8,1.2,2,3,5])
TPOT = Histogram("inference_tpot_seconds", "Time per output token",
                 ["model"], buckets=[.005,.01,.02,.03,.05,.08,.12,.2])
QUEUE = Histogram("inference_queue_seconds", "Time waiting before compute",
                  ["model"], buckets=[.001,.005,.01,.05,.1,.5,1,5])
REQS = Counter("inference_requests_total", "Requests", ["model", "status"])
TOKENS = Counter("inference_tokens_total", "Tokens", ["model", "kind"])
RUNNING = Gauge("inference_running_requests", "In-flight requests", ["model"])
KV = Gauge("inference_kv_cache_utilisation", "KV-cache used fraction", ["model"])

app = FastAPI()
app.mount("/metrics", make_asgi_app())

## Gotchas & Pitfalls

- **Averaging percentiles.** Example 2. Always `sum(rate(..._bucket[5m])) by (le)` and
  then `histogram_quantile`. If a dashboard shows `avg(p99)`, it is wrong.
- **Using a Summary instead of a Histogram.** Summaries compute quantiles inside each
  process and cannot be aggregated across replicas at all. Use histograms for anything
  that runs more than once.
- **Label cardinality explosions.** A `request_id`, `user_id`, `prompt` or raw `path`
  label creates one time series per distinct value and will take down your Prometheus.
  Labels must be bounded and low-cardinality: model name, status, maybe tenant tier.
- **Measuring at the wrong boundary.** Timing only the model call excludes queue time,
  which is usually where the latency went. Start the clock when the request *arrives*.
- **Default histogram buckets.** Example 3. They are generic web-service buckets and are
  usually wrong for inference, especially for TPOT, which lives in single-digit
  milliseconds.
- **Treating GPU utilisation as saturation.** `DCGM_FI_DEV_GPU_UTIL` reports whether a
  kernel was resident, not whether the GPU was doing useful work. It can read 100% while
  the device is memory-bandwidth-starved. Pair it with achieved occupancy, KV-cache
  utilisation and queue depth.
- **Alerting on raw latency without a token normaliser.** Longer user prompts raise
  end-to-end latency legitimately. Alert on TTFT and TPOT, which are load signals, not
  demand signals.
- **Scrape interval versus incident duration.** A 60s scrape cannot see a 10s stall. For
  latency-critical services scrape at 10–15s and be aware the tail you plot is already
  smoothed.
- **Counter resets on redeploy.** Always `rate()`/`increase()` counters — reading the raw
  value across a restart gives nonsense. This is why counters must only increase.
- **No exemplars.** Without trace IDs on your histogram buckets you can see *that* the
  p99 is bad and never find a single example of it. Exemplars are the bridge from metrics
  to traces.

## When to Use vs Alternatives

| Question | Tool |
|---|---|
| Is the fleet healthy? Is it getting worse? | **Prometheus metrics** (this notebook) — cheap, aggregated, alertable |
| Why was *this specific* request slow? | **Distributed tracing** — OpenTelemetry, Jaeger, Tempo. Metrics cannot answer this |
| What exactly happened at 03:14? | **Structured logs** — expensive at volume; sample them |
| Is the GPU the bottleneck? | [**DCGM Exporter**](../nvidia-gpu-components/dcgm-exporter.ipynb) + Nsight for kernel-level work |
| Are the model's *outputs* getting worse? | ML monitoring — drift detection, [MLflow](../../02-ai-ml-tooling/mlflow.ipynb) evaluation, human review. Latency metrics say nothing about quality |

**The honest scope.** Metrics tell you *that* something is wrong and roughly where;
traces tell you *why* for a given request; logs tell you *what exactly* happened. Trying
to make metrics do the second and third jobs is what produces the cardinality explosions
above. Instrument all three, and keep the boundary clear.

The category this notebook does **not** cover, and which matters just as much for
inference, is **output quality monitoring** — a model that has silently started returning
worse answers looks perfectly healthy on every metric here. Latency and correctness are
independent axes, and only one of them is visible to Prometheus.

Most serving frameworks emit a good deal of this already: vLLM, TGI and Triton all expose
`/metrics` with queue, batch and cache statistics. Check what your server already gives
you before instrumenting by hand — the useful work is usually choosing buckets, fixing
the aggregation, and adding the queue boundary, not writing counters from scratch.

## Resources

- [Prometheus: metric types](https://prometheus.io/docs/concepts/metric_types/) — the histogram-versus-summary distinction that Example 2 turns on.
- [Prometheus: histograms and summaries best practices](https://prometheus.io/docs/practices/histograms/) — including the `histogram_quantile` interpolation used in Example 3.
- [The RED Method](https://grafana.com/blog/2018/08/02/the-red-method-how-to-instrument-your-services/) — Rate, Errors, Duration, and why it is the right default for request-driven services.
- [Brendan Gregg: the USE Method](https://www.brendangregg.com/usemethod.html) — the resource-side complement, and the source of the utilisation-versus-saturation distinction.
- [vLLM production metrics](https://docs.vllm.ai/en/latest/usage/metrics.html) — the exact metric names a modern LLM server exposes, including KV-cache and preemption counters.
- [NVIDIA DCGM Exporter](https://github.com/NVIDIA/dcgm-exporter) — GPU metrics into Prometheus; pairs with [the DCGM notebook](../nvidia-gpu-components/dcgm-exporter.ipynb).
- [OpenTelemetry: metrics vs traces vs logs](https://opentelemetry.io/docs/concepts/signals/) — where the boundary between the three signals should sit.
- [Google SRE Workbook: Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/) — burn-rate alerting, and why static latency thresholds page you at the wrong times.

Your inference endpoint's p99 latency has doubled. Which single exported metric would most quickly tell you whether to add replicas or to optimise the model?

Using the kitchen analogy, explain the difference between utilisation and saturation for an inference server, and say which one you should alert on.

Why should a replicated inference service export latency as a Prometheus Histogram rather than a Summary?

In [ ]:
def histogram_quantile(counts, bounds, q):
    ...


In [ ]:
counts, bounds = [10, 10, 10, 0], [1.0, 2.0, 3.0]
assert abs(histogram_quantile(counts, bounds, 0.5) - 1.5) < 1e-9, \
    histogram_quantile(counts, bounds, 0.5)
assert abs(histogram_quantile(counts, bounds, 0.0) - 0.0) < 1e-9
prev = -1.0
for q in (0.1, 0.25, 0.5, 0.75, 0.9, 0.99):
    v = histogram_quantile(counts, bounds, q)
    assert v >= prev, 'quantiles must be monotonic in q'
    prev = v
assert histogram_quantile([0, 0, 0, 0], bounds, 0.5) == 0.0
one = histogram_quantile([100, 0, 0, 0], bounds, 0.99)
assert 0.0 <= one <= 1.0, one


Which label would you refuse to attach to an inference latency metric, and why?

Metrics, traces and logs each answer a different question. State which question each one answers, and name the category of inference problem that none of the three will catch.